# Faceoff Decay Analysis — Zone-Separated (Phase 2, Area 3)

**Goal:** Determine how shot quality decays as time elapses after a faceoff, treating each faceoff zone as a semantically distinct event.

**Core premise:** Offensive (O), defensive (D), and neutral (N) zone faceoffs produce fundamentally different tactical situations. An offensive-zone faceoff is a direct scoring opportunity; a defensive-zone faceoff is primarily a survival event; a neutral-zone faceoff is transitional. Aggregating them obscures zone-specific decay patterns and inflection points.

**Key questions:**
1. Do the proposed recency bin boundaries (5/15/30/60s) match inflection points *within each zone*?
2. How do decay shape, shot volume, and goal rate differ across O/D/N zones?
3. Does manpower state interact differently with faceoff decay in each zone?
4. Is a continuous representation (e.g., `log(seconds + 1)`) better than bins, and does the answer vary by zone?

**Data source:** `shot_events` table in `data/nhl_data.db`

In [ ]:
import os
import sys
import sqlite3

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

# Add src/ to path — handle CWD being project root or notebooks/
for _candidate in [os.path.join(os.getcwd(), "src"),
                   os.path.join(os.getcwd(), "..", "src")]:
    _candidate = os.path.abspath(_candidate)
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

from database import DATABASE_PATH

sns.set_theme(style="whitegrid")
print(f"Database: {DATABASE_PATH}")
conn = sqlite3.connect(DATABASE_PATH)
conn.row_factory = sqlite3.Row
print("Connected.")

## 1. Data availability check

Verify we have enough shot events with `seconds_since_faceoff` populated, broken down by faceoff zone.

In [ ]:
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM shot_events")
total_shots = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM shot_events WHERE seconds_since_faceoff IS NOT NULL")
with_faceoff = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM shot_events WHERE faceoff_zone_code IS NOT NULL")
with_zone = cur.fetchone()[0]

coverage_pct = (with_faceoff / total_shots * 100) if total_shots > 0 else 0
zone_pct = (with_zone / total_shots * 100) if total_shots > 0 else 0

print(f"Total shot events:               {total_shots:,}")
print(f"With seconds_since_faceoff:       {with_faceoff:,} ({coverage_pct:.1f}%)")
print(f"With faceoff_zone_code:           {with_zone:,} ({zone_pct:.1f}%)")
print()

ZONE_CODES = ["O", "D", "N"]
ZONE_LABELS = {"O": "Offensive zone", "D": "Defensive zone", "N": "Neutral zone"}

cur.execute("""
    SELECT faceoff_zone_code,
           COUNT(*) AS shots,
           SUM(is_goal) AS goals,
           ROUND(CAST(SUM(is_goal) AS REAL) / COUNT(*), 4) AS goal_rate
    FROM shot_events
    WHERE seconds_since_faceoff IS NOT NULL
      AND faceoff_zone_code IS NOT NULL
    GROUP BY faceoff_zone_code
    ORDER BY faceoff_zone_code
""")
print(f"{'Zone':<20} {'Shots':>10} {'Goals':>8} {'Goal Rate':>10}")
print("-" * 52)
for row in cur.fetchall():
    label = ZONE_LABELS.get(row[0], row[0])
    print(f"{label:<20} {row[1]:>10,} {row[2]:>8,} {row[3]:>10.4f}")

if total_shots == 0:
    print("\n*** No shot events found. Run the scraper first. ***")

## 2. Per-zone decay curves: shot volume and goal rate

Each faceoff zone is a distinct event. We plot shot volume (left) and goal rate (right) for each zone separately, so that zone-specific inflection points are visible without being averaged away.

- **Offensive zone (O):** Direct scoring opportunity — expect a sharp early spike.
- **Defensive zone (D):** Survival event — expect lower volume and flatter goal rate.
- **Neutral zone (N):** Transitional — expect intermediate patterns.

In [ ]:
MAX_SECONDS = 120
SMOOTH_WINDOW = 5

BIN_BOUNDARIES = [5, 15, 30, 60]
BIN_COLORS = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71"]

ZONE_COLORS = {"O": "#e74c3c", "D": "#3498db", "N": "#2ecc71"}

zone_data = {}

for zone in ZONE_CODES:
    cur.execute("""
        SELECT seconds_since_faceoff AS sec,
               COUNT(*)              AS shot_count,
               SUM(is_goal)          AS goal_count
        FROM shot_events
        WHERE seconds_since_faceoff IS NOT NULL
          AND seconds_since_faceoff BETWEEN 0 AND ?
          AND faceoff_zone_code = ?
        GROUP BY seconds_since_faceoff
        ORDER BY seconds_since_faceoff
    """, (MAX_SECONDS, zone))

    rows = cur.fetchall()
    if not rows:
        continue

    z_sec = np.array([r[0] for r in rows])
    z_shots = np.array([r[1] for r in rows])
    z_goals = np.array([r[2] for r in rows])
    z_rate = np.where(z_shots > 0, z_goals / z_shots, 0.0)

    if len(z_rate) >= SMOOTH_WINDOW:
        z_smooth = np.convolve(z_rate, np.ones(SMOOTH_WINDOW) / SMOOTH_WINDOW, mode="same")
    else:
        z_smooth = z_rate

    zone_data[zone] = {
        "seconds": z_sec, "shots": z_shots, "goals": z_goals,
        "goal_rate": z_rate, "smoothed_rate": z_smooth,
    }

fig, axes = plt.subplots(len(zone_data), 2, figsize=(16, 5 * len(zone_data)), sharex=True)
if len(zone_data) == 1:
    axes = axes.reshape(1, -1)

for row_idx, zone in enumerate(ZONE_CODES):
    if zone not in zone_data:
        continue
    zd = zone_data[zone]
    color = ZONE_COLORS[zone]
    label = ZONE_LABELS[zone]

    ax_vol = axes[row_idx, 0]
    ax_vol.bar(zd["seconds"], zd["shots"], width=1.0, color=color, alpha=0.7)
    ax_vol.set_ylabel("Shot count")
    ax_vol.set_title(f"{label} — Shot Volume")
    for b in BIN_BOUNDARIES:
        ax_vol.axvline(x=b + 0.5, color="#bdc3c7", linestyle="--", linewidth=1.2)

    ax_gr = axes[row_idx, 1]
    ax_gr.plot(zd["seconds"], zd["goal_rate"], alpha=0.25, color="#95a5a6", linewidth=0.8, label="Raw (1s)")
    ax_gr.plot(zd["seconds"], zd["smoothed_rate"], color=color, linewidth=2.0,
               label=f"Smoothed ({SMOOTH_WINDOW}s avg)")
    ax_gr.set_ylabel("Goal rate")
    ax_gr.set_title(f"{label} — Goal Rate")
    ax_gr.legend(fontsize=8)
    for b in BIN_BOUNDARIES:
        ax_gr.axvline(x=b + 0.5, color="#bdc3c7", linestyle="--", linewidth=1.2)

axes[-1, 0].set_xlabel("Seconds since faceoff")
axes[-1, 1].set_xlabel("Seconds since faceoff")

fig.tight_layout()
plt.show()

for zone in ZONE_CODES:
    if zone not in zone_data:
        continue
    zd = zone_data[zone]
    print(f"{ZONE_LABELS[zone]}: {zd['shots'].sum():,} shots, {zd['goals'].sum():,} goals "
          f"(goal rate {zd['goals'].sum() / zd['shots'].sum():.4f})")

In [ ]:
# Normalized overlay: compare zones on the same scale
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for zone in ZONE_CODES:
    if zone not in zone_data:
        continue
    zd = zone_data[zone]
    color = ZONE_COLORS[zone]
    label = ZONE_LABELS[zone]

    # Normalize shot volume to peak=1 within each zone
    norm_shots = zd["shots"] / zd["shots"].max() if zd["shots"].max() > 0 else zd["shots"]
    ax1.plot(zd["seconds"], norm_shots, color=color, linewidth=1.5, alpha=0.8, label=label)

    ax2.plot(zd["seconds"], zd["smoothed_rate"], color=color, linewidth=2.0, label=label)

ax1.set_xlabel("Seconds since faceoff")
ax1.set_ylabel("Normalized shot volume (peak = 1)")
ax1.set_title("Decay Shape Comparison — Normalized Shot Volume")
ax1.legend()
for b in BIN_BOUNDARIES:
    ax1.axvline(x=b + 0.5, color="#bdc3c7", linestyle=":", linewidth=1.0)

ax2.set_xlabel("Seconds since faceoff")
ax2.set_ylabel("Goal rate (smoothed)")
ax2.set_title("Goal Rate Comparison Across Zones")
ax2.legend()
for b in BIN_BOUNDARIES:
    ax2.axvline(x=b + 0.5, color="#bdc3c7", linestyle=":", linewidth=1.0)

fig.tight_layout()
plt.show()

## 3. Per-zone recency bin statistics

Summarize shot volume, goal rate, and average distance within each proposed bin — separately for each zone. This reveals whether the same bin boundaries are appropriate across all three zones.

In [ ]:
for zone in ZONE_CODES:
    label = ZONE_LABELS[zone]
    print(f"\n{'='*68}")
    print(f"  {label}")
    print(f"{'='*68}")

    cur.execute("""
        SELECT
            CASE
                WHEN seconds_since_faceoff BETWEEN 0 AND 5   THEN 'immediate (0-5s)'
                WHEN seconds_since_faceoff BETWEEN 6 AND 15  THEN 'early (6-15s)'
                WHEN seconds_since_faceoff BETWEEN 16 AND 30 THEN 'mid (16-30s)'
                WHEN seconds_since_faceoff BETWEEN 31 AND 60 THEN 'late (31-60s)'
                ELSE 'steady_state (61+s)'
            END AS recency_bin,
            COUNT(*)                                           AS shots,
            SUM(is_goal)                                       AS goals,
            ROUND(CAST(SUM(is_goal) AS REAL) / COUNT(*), 4)    AS goal_rate,
            ROUND(AVG(distance_to_goal), 1)                    AS avg_distance
        FROM shot_events
        WHERE seconds_since_faceoff IS NOT NULL
          AND seconds_since_faceoff >= 0
          AND faceoff_zone_code = ?
        GROUP BY recency_bin
        ORDER BY
            CASE recency_bin
                WHEN 'immediate (0-5s)' THEN 1
                WHEN 'early (6-15s)' THEN 2
                WHEN 'mid (16-30s)' THEN 3
                WHEN 'late (31-60s)' THEN 4
                ELSE 5
            END
    """, (zone,))

    print(f"{'Recency Bin':<25} {'Shots':>10} {'Goals':>8} {'Goal Rate':>10} {'Avg Dist':>10}")
    print("-" * 68)
    for row in cur.fetchall():
        print(f"{row[0]:<25} {row[1]:>10,} {row[2]:>8,} {row[3]:>10.4f} {row[4]:>10.1f}")

## 4. Cross-zone bin comparison

Side-by-side bar chart comparing goal rate across zones within each recency bin, highlighting how the same time window means different things for different zones.

In [ ]:
BIN_NAMES = ["immediate\n(0-5s)", "early\n(6-15s)", "mid\n(16-30s)", "late\n(31-60s)", "steady_state\n(61+s)"]
BIN_SQL_LABELS = ["immediate (0-5s)", "early (6-15s)", "mid (16-30s)", "late (31-60s)", "steady_state (61+s)"]

cur.execute("""
    SELECT
        faceoff_zone_code AS zone,
        CASE
            WHEN seconds_since_faceoff BETWEEN 0 AND 5   THEN 'immediate (0-5s)'
            WHEN seconds_since_faceoff BETWEEN 6 AND 15  THEN 'early (6-15s)'
            WHEN seconds_since_faceoff BETWEEN 16 AND 30 THEN 'mid (16-30s)'
            WHEN seconds_since_faceoff BETWEEN 31 AND 60 THEN 'late (31-60s)'
            ELSE 'steady_state (61+s)'
        END AS recency_bin,
        CAST(SUM(is_goal) AS REAL) / COUNT(*) AS goal_rate
    FROM shot_events
    WHERE seconds_since_faceoff IS NOT NULL
      AND seconds_since_faceoff >= 0
      AND faceoff_zone_code IS NOT NULL
    GROUP BY zone, recency_bin
""")

zone_bin_rates = {}
for row in cur.fetchall():
    zone_bin_rates[(row[0], row[1])] = row[2]

x = np.arange(len(BIN_SQL_LABELS))
BAR_WIDTH = 0.25

fig, ax = plt.subplots(figsize=(14, 6))

for i, zone in enumerate(ZONE_CODES):
    rates = [zone_bin_rates.get((zone, b), 0.0) for b in BIN_SQL_LABELS]
    ax.bar(x + i * BAR_WIDTH, rates, BAR_WIDTH,
           color=ZONE_COLORS[zone], alpha=0.8, label=ZONE_LABELS[zone])

ax.set_xticks(x + BAR_WIDTH)
ax.set_xticklabels(BIN_NAMES)
ax.set_ylabel("Goal rate")
ax.set_title("Goal Rate by Recency Bin — Separated by Faceoff Zone")
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=1))

fig.tight_layout()
plt.show()

## 5. Zone convergence analysis

At what point do zones stop being distinct? If all three zones converge to similar goal rates after N seconds, that marks where faceoff zone context stops mattering.

In [ ]:
# Compute the spread (max - min) of smoothed goal rates across zones at each second
CONVERGENCE_SMOOTH = 10  # wider window for clearer convergence signal

# Re-smooth with wider window for convergence analysis
zone_wide_smooth = {}
common_seconds = None

for zone in ZONE_CODES:
    if zone not in zone_data:
        continue
    zd = zone_data[zone]
    if len(zd["goal_rate"]) >= CONVERGENCE_SMOOTH:
        ws = np.convolve(zd["goal_rate"],
                         np.ones(CONVERGENCE_SMOOTH) / CONVERGENCE_SMOOTH, mode="same")
    else:
        ws = zd["goal_rate"]
    zone_wide_smooth[zone] = (zd["seconds"], ws)
    if common_seconds is None:
        common_seconds = set(zd["seconds"])
    else:
        common_seconds &= set(zd["seconds"])

if common_seconds and len(zone_wide_smooth) == len(ZONE_CODES):
    common_seconds = sorted(common_seconds)
    cs_arr = np.array(common_seconds)

    # Build rate arrays aligned to common seconds
    zone_rates_aligned = {}
    for zone in ZONE_CODES:
        sec, ws = zone_wide_smooth[zone]
        sec_to_rate = dict(zip(sec, ws))
        zone_rates_aligned[zone] = np.array([sec_to_rate[s] for s in common_seconds])

    # Spread = max zone rate - min zone rate at each second
    all_rates = np.stack([zone_rates_aligned[z] for z in ZONE_CODES])
    spread = all_rates.max(axis=0) - all_rates.min(axis=0)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Overlay smoothed rates
    for zone in ZONE_CODES:
        ax1.plot(cs_arr, zone_rates_aligned[zone], color=ZONE_COLORS[zone],
                 linewidth=2.0, label=ZONE_LABELS[zone])
    ax1.set_ylabel(f"Goal rate ({CONVERGENCE_SMOOTH}s smoothed)")
    ax1.set_title("Zone-Separated Goal Rates — Convergence View")
    ax1.legend()

    # Spread plot
    ax2.fill_between(cs_arr, spread, alpha=0.4, color="#8e44ad")
    ax2.plot(cs_arr, spread, color="#8e44ad", linewidth=1.5)
    ax2.set_ylabel("Goal rate spread (max − min)")
    ax2.set_xlabel("Seconds since faceoff")
    ax2.set_title("Cross-Zone Goal Rate Spread Over Time")

    for ax in (ax1, ax2):
        for b in BIN_BOUNDARIES:
            ax.axvline(x=b + 0.5, color="#bdc3c7", linestyle=":", linewidth=1.0)

    fig.tight_layout()
    plt.show()

    # Find approximate convergence point (spread < 20% of initial spread)
    CONVERGENCE_THRESHOLD = 0.2
    initial_spread = spread[:5].mean() if len(spread) >= 5 else spread[0]
    threshold = initial_spread * CONVERGENCE_THRESHOLD
    converge_mask = spread < threshold
    if converge_mask.any():
        converge_sec = cs_arr[converge_mask][0]
        print(f"Zones converge (spread < {CONVERGENCE_THRESHOLD:.0%} of initial) at ~{converge_sec}s")
    else:
        print("Zones do not fully converge within the analysis window.")
else:
    print("Insufficient zone data for convergence analysis.")

## 6. Zone × manpower state interaction

Does manpower state interact differently with faceoff decay depending on zone? Each manpower state (5v5, 5v4, 4v5, etc.) is a distinct tactical situation and is analyzed separately rather than grouped. Low-volume states are included but labeled with sample size for context.

In [ ]:
cur.execute("""
    SELECT manpower_state, COUNT(*) AS n
    FROM shot_events
    WHERE seconds_since_faceoff IS NOT NULL
      AND manpower_state IS NOT NULL
    GROUP BY manpower_state
    ORDER BY n DESC
""")
all_mp_states = [(row[0], row[1]) for row in cur.fetchall()]

MIN_SHOTS_FOR_CURVE = 50

_MP_PALETTE = sns.color_palette("tab10", n_colors=len(all_mp_states))
MP_STATE_COLORS = {state: _MP_PALETTE[i] for i, (state, _) in enumerate(all_mp_states)}

print(f"Manpower states found: {len(all_mp_states)}")
for state, n in all_mp_states:
    marker = "" if n >= MIN_SHOTS_FOR_CURVE else " (too few for curve)"
    print(f"  {state}: {n:,} shots{marker}")

fig, axes = plt.subplots(len(ZONE_CODES), 2, figsize=(16, 6 * len(ZONE_CODES)), sharex=True)

for row_idx, zone in enumerate(ZONE_CODES):
    ax_vol = axes[row_idx, 0]
    ax_gr = axes[row_idx, 1]
    zone_label = ZONE_LABELS[zone]

    for mp_state, total_n in all_mp_states:
        cur.execute("""
            SELECT seconds_since_faceoff,
                   COUNT(*) AS shots,
                   SUM(is_goal) AS goals
            FROM shot_events
            WHERE seconds_since_faceoff IS NOT NULL
              AND seconds_since_faceoff BETWEEN 0 AND ?
              AND faceoff_zone_code = ?
              AND manpower_state = ?
            GROUP BY seconds_since_faceoff
            ORDER BY seconds_since_faceoff
        """, (MAX_SECONDS, zone, mp_state))

        rows = cur.fetchall()
        if not rows:
            continue

        m_sec = np.array([r[0] for r in rows])
        m_shots = np.array([r[1] for r in rows])
        m_goals = np.array([r[2] for r in rows])
        zone_mp_total = m_shots.sum()

        if zone_mp_total < MIN_SHOTS_FOR_CURVE:
            continue

        m_rate = np.where(m_shots > 0, m_goals / m_shots, 0.0)

        if len(m_rate) >= SMOOTH_WINDOW:
            m_smooth = np.convolve(m_rate, np.ones(SMOOTH_WINDOW) / SMOOTH_WINDOW, mode="same")
        else:
            m_smooth = m_rate

        color = MP_STATE_COLORS[mp_state]
        curve_label = f"{mp_state} (n={zone_mp_total:,})"
        ax_vol.plot(m_sec, m_shots, color=color, alpha=0.8, linewidth=1.5, label=curve_label)
        ax_gr.plot(m_sec, m_smooth, color=color, linewidth=2.0, label=curve_label)

    ax_vol.set_ylabel("Shot count")
    ax_vol.set_title(f"{zone_label} — Shot Volume by Manpower State")
    ax_vol.legend(fontsize=7, loc="upper right")

    ax_gr.set_ylabel("Goal rate (smoothed)")
    ax_gr.set_title(f"{zone_label} — Goal Rate by Manpower State")
    ax_gr.legend(fontsize=7, loc="upper right")

    for ax in (ax_vol, ax_gr):
        for b in BIN_BOUNDARIES:
            ax.axvline(x=b + 0.5, color="#bdc3c7", linestyle=":", linewidth=1.0)

axes[-1, 0].set_xlabel("Seconds since faceoff")
axes[-1, 1].set_xlabel("Seconds since faceoff")

fig.tight_layout()
plt.show()

## 7. Per-zone exponential decay fit

Fit an exponential decay to each zone separately. The decay rate and R-squared tell us whether zones have genuinely different half-lives. If offensive-zone shots decay faster, the faceoff context window is shorter there — which has implications for bin boundary selection.

In [ ]:
fig, axes = plt.subplots(len(ZONE_CODES), 3, figsize=(18, 5 * len(ZONE_CODES)))

fit_results = {}

for row_idx, zone in enumerate(ZONE_CODES):
    if zone not in zone_data:
        continue

    zd = zone_data[zone]
    z_sec = zd["seconds"]
    z_shots = zd["shots"]
    z_rate = zd["goal_rate"]
    label = ZONE_LABELS[zone]
    color = ZONE_COLORS[zone]

    if len(z_shots) == 0 or z_shots.max() == 0:
        continue

    normalized = z_shots / z_shots.max()
    log_seconds = np.log1p(z_sec)

    valid = normalized > 0
    if valid.sum() <= 2:
        continue

    log_counts = np.log(normalized[valid])
    sec_valid = z_sec[valid]

    coeffs = np.polyfit(sec_valid, log_counts, 1)
    decay_rate = coeffs[0]
    amplitude = np.exp(coeffs[1])
    exp_fit = amplitude * np.exp(decay_rate * z_sec)

    residuals = normalized - exp_fit
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((normalized - normalized.mean()) ** 2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    half_life = -np.log(2) / decay_rate if decay_rate < 0 else float("inf")

    fit_results[zone] = {
        "decay_rate": decay_rate, "r_squared": r_squared, "half_life": half_life,
    }

    # Panel 1: Raw + exponential fit
    ax1 = axes[row_idx, 0]
    ax1.bar(z_sec, normalized, width=1.0, alpha=0.5, color=color, label="Observed")
    ax1.plot(z_sec, exp_fit, color="black", linewidth=2.0,
             label=f"Exp fit (λ={decay_rate:.4f}/s)")
    ax1.set_ylabel("Normalized shot count")
    ax1.set_title(f"{label} — Exponential Decay Fit")
    ax1.legend(fontsize=8)

    # Panel 2: Log(seconds+1) vs goal rate
    ax2 = axes[row_idx, 1]
    ax2.scatter(log_seconds, z_rate, alpha=0.3, s=8, color=color)
    if len(z_rate) >= SMOOTH_WINDOW:
        sort_idx = np.argsort(log_seconds)
        ax2.plot(log_seconds[sort_idx], zd["smoothed_rate"][sort_idx],
                 color="black", linewidth=2.0, label="Smoothed")
    ax2.set_xlabel("log(seconds + 1)")
    ax2.set_ylabel("Goal rate")
    ax2.set_title(f"{label} — Goal Rate vs Log Time")
    ax2.legend(fontsize=8)

    # Panel 3: Residuals
    ax3 = axes[row_idx, 2]
    ax3.bar(z_sec, residuals, width=1.0, alpha=0.6, color=color)
    ax3.axhline(y=0, color="black", linewidth=0.5)
    ax3.set_xlabel("Seconds since faceoff")
    ax3.set_ylabel("Residual")
    ax3.set_title(f"{label} — Fit Residuals")

fig.tight_layout()
plt.show()

# Summary table
print(f"\n{'Zone':<20} {'Decay Rate':>12} {'Half-life':>12} {'R²':>8}")
print("-" * 56)
for zone in ZONE_CODES:
    if zone not in fit_results:
        continue
    fr = fit_results[zone]
    print(f"{ZONE_LABELS[zone]:<20} {fr['decay_rate']:>12.5f} {fr['half_life']:>10.1f}s {fr['r_squared']:>8.4f}")

## 8. Per-zone shot distance over time

Shot distance after a faceoff should differ sharply by zone. Offensive-zone faceoffs start close to the net; defensive-zone faceoffs start far away. How quickly does each zone's average shot distance converge to the league-wide mean?

In [ ]:
fig, axes = plt.subplots(1, len(ZONE_CODES), figsize=(18, 5), sharey=True)

cur.execute("""
    SELECT AVG(distance_to_goal)
    FROM shot_events
    WHERE distance_to_goal IS NOT NULL
""")
league_avg_dist = cur.fetchone()[0]

for col_idx, zone in enumerate(ZONE_CODES):
    ax = axes[col_idx]

    cur.execute("""
        SELECT seconds_since_faceoff,
               AVG(distance_to_goal) AS avg_dist,
               COUNT(*) AS n
        FROM shot_events
        WHERE seconds_since_faceoff IS NOT NULL
          AND seconds_since_faceoff BETWEEN 0 AND ?
          AND distance_to_goal IS NOT NULL
          AND faceoff_zone_code = ?
        GROUP BY seconds_since_faceoff
        ORDER BY seconds_since_faceoff
    """, (MAX_SECONDS, zone))

    rows = cur.fetchall()
    if not rows:
        continue

    d_sec = np.array([r[0] for r in rows])
    d_avg = np.array([r[1] for r in rows])

    if len(d_avg) >= SMOOTH_WINDOW:
        d_smooth = np.convolve(d_avg, np.ones(SMOOTH_WINDOW) / SMOOTH_WINDOW, mode="same")
    else:
        d_smooth = d_avg

    color = ZONE_COLORS[zone]
    ax.scatter(d_sec, d_avg, alpha=0.2, s=8, color="#95a5a6")
    ax.plot(d_sec, d_smooth, color=color, linewidth=2.0, label=f"Smoothed ({SMOOTH_WINDOW}s)")
    if league_avg_dist is not None:
        ax.axhline(y=league_avg_dist, color="#e67e22", linestyle="--", linewidth=1.2,
                   label=f"League avg ({league_avg_dist:.1f} ft)")
    ax.set_xlabel("Seconds since faceoff")
    ax.set_title(f"{ZONE_LABELS[zone]} — Avg Shot Distance")
    ax.legend(fontsize=8)

    for b in BIN_BOUNDARIES:
        ax.axvline(x=b + 0.5, color="#bdc3c7", linestyle=":", linewidth=1.0)

axes[0].set_ylabel("Average distance to goal (ft)")

fig.tight_layout()
plt.show()

## 9. Summary and recommendations

**Interpret the zone-separated plots above to answer:**

1. **Zone-specific bin boundaries:** Do the 5/15/30/60s boundaries align with inflection points *within each zone*? Offensive-zone faceoffs may need a tighter initial window (e.g., 3s instead of 5s) if the spike is sharper. Defensive-zone faceoffs may show a flatter curve where the 5s "immediate" bin captures no meaningful signal. Consider defining zone-specific bin boundaries in `_FACEOFF_RECENCY_BINS` if decay shapes differ substantially.

2. **Decay rate differences:** Compare the per-zone exponential fit half-lives. If offensive-zone half-life is significantly shorter than defensive-zone, this confirms that faceoff zone and recency interact — a key justification for zone-specific features rather than a single pooled decay model.

3. **Convergence point:** The zone convergence analysis (section 5) identifies where faceoff zone context stops adding information. Beyond this point, zone-specific features add noise, not signal. Use this to set the upper bound of the faceoff recency window.

4. **Manpower × zone interaction:** If PP faceoffs in the offensive zone show a dramatically different decay pattern from PP faceoffs in the defensive zone (section 6), consider a three-way interaction feature: `zone × manpower × recency`.

5. **Distance patterns:** The per-zone distance curves (section 8) explain *why* zones produce different goal rates. Offensive-zone faceoffs generate closer shots initially; the distance advantage decays as play transitions. This mechanistic understanding validates treating zones as distinct events.

6. **Continuous vs. binned:** Compare per-zone R-squared values. If the exponential fit quality varies by zone, the optimal representation may differ: bins for zones with step-function breaks, `log(seconds + 1)` for zones with smooth decay. A pragmatic approach: include `zone × log(seconds + 1)` as zone-specific continuous features alongside zone-specific bins, and let the model select.

In [ ]:
conn.close()
print("Done.")